In [2]:
!pip -q install ultralytics

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.0 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
DATA_DIR = "/content/drive/MyDrive/augmentation2_code"

In [4]:
from pathlib import Path

data_dir = Path(DATA_DIR)
exts = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}

for split in ["train", "val", "test"]:
    img_dir = data_dir / "images" / split
    files = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in exts])
    (data_dir / f"{split}.txt").write_text("\n".join(str(p) for p in files) + ("\n" if files else ""))

print("Rebuilt train/val/test txt files.")

Rebuilt train/val/test txt files.


In [5]:
yaml_text = f"""path: {DATA_DIR}
train: train.txt
val: val.txt
test: test.txt
names:
  0: page_left
  1: page_right
"""
with open("/content/aug2_data.yaml", "w") as f:
    f.write(yaml_text)

print("Wrote /content/aug2_data.yaml")

Wrote /content/aug2_data.yaml


In [7]:
from pathlib import Path
from PIL import Image

DATA_DIR = Path("/content/drive/MyDrive/augmentation2_code")
exts = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}

converted = 0
for split in ["train", "val", "test"]:
    for p in (DATA_DIR / "images" / split).iterdir():
        if p.suffix.lower() not in exts:
            continue
        with Image.open(p) as im:
            if im.mode != "RGB":
                im.convert("RGB").save(p)
                converted += 1

print("Converted to RGB:", converted)

Converted to RGB: 546


In [8]:
!yolo task=segment mode=train \
  model=yolov8s-seg.pt \
  data=/content/aug2_data.yaml \
  imgsz=640 batch=8 epochs=50 device=0 \
  name=aug2_yolov8s

Ultralytics 8.4.34 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/aug2_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aug2_yolov8s2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspect

In [9]:
!yolo task=segment mode=val \
  model=/content/runs/segment/aug2_yolov8s2/weights/best.pt \
  data=/content/aug2_data.yaml \
  split=test device=0

Ultralytics 8.4.34 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,780,374 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 641.0±358.9 MB/s, size: 4580.5 KB)
val: Scanning /content/drive/MyDrive/augmentation2_code/labels/test... 142 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 142/142 6.3it/s 22.4s
val: New cache created: /content/drive/MyDrive/augmentation2_code/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.0s/it 9.2s
                   all        142        284          1          1      0.995      0.989          1          1      0.995      0.992
             page_left        142        142          1          1      0.995      0.989          1          1      0.995      0.994
            page_right        142        142          1          1      0.9

In [10]:
!yolo task=segment mode=predict \
  model=/content/runs/segment/aug2_yolov8s2/weights/best.pt \
  source=/content/drive/MyDrive/augmentation2_code/images/test \
  conf=0.25 save=True device=0

Ultralytics 8.4.34 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,780,374 parameters, 0 gradients, 39.9 GFLOPs

image 1/142 /content/drive/MyDrive/augmentation2_code/images/test/na121_al-moqatam_1908_oct-dec_00005.tif: 448x640 1 page_left, 1 page_right, 94.2ms
image 2/142 /content/drive/MyDrive/augmentation2_code/images/test/na121_al-moqatam_1908_oct-dec_00005_rot90.tif: 640x448 1 page_left, 1 page_right, 81.9ms
image 3/142 /content/drive/MyDrive/augmentation2_code/images/test/na121_al-moqatam_1908_oct-dec_00014.tif: 480x640 1 page_left, 1 page_right, 82.6ms
image 4/142 /content/drive/MyDrive/augmentation2_code/images/test/na121_al-moqatam_1908_oct-dec_00014_rot90.tif: 640x480 1 page_left, 1 page_right, 89.6ms
image 5/142 /content/drive/MyDrive/augmentation2_code/images/test/na121_al-moqatam_1908_oct-dec_00015.tif: 448x640 1 page_left, 1 page_right, 9.1ms
image 6/142 /content/drive/MyDrive/augmentation2_code/images/test/na121

In [30]:
from pathlib import Path
for p in Path("/content").rglob("best.pt"):
    print(p)

/content/runs/segment/aug2_yolov8s2/weights/best.pt
